# Test ASEGtoHDF

In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
import pathlib
from scipy.interpolate import CloughTocher2DInterpolator
from scipy import interpolate
import filebrowser as fb
import aseg_gdf2 as aseg

groupName = "Test AsegGDF"
projectName = "dummy"

In [2]:
local_docs = "/Users/markdransfield/"
data_root = local_docs + "Documents/GitHub/AirGravQC/examples/Airmag_GDF/"

muppetGDF_file = Path(data_root + r'Example_AeroMag_MuppetTown_2009.dfn')
muppetHDF_file = muppetGDF_file.with_suffix('.hdf5')

#### Define the functions

`channelNames, channelsOut, lineIdx, flightIdx, dateIdx, zoneIdx = get_gdfchannames(GDF_file, omitChannels=['RT'])`

In [192]:
def get_gdfchannames(GDF_file, omitChannels=['RT']):
    gdf = aseg.read(GDF_file, method='fixed-widths')#, engine='pandas')
    channelNames = gdf.field_names()
    channelsOut = channelNames # these are the channels we will save
        
    # next, identify the column containing line numbers etc
    # we won't save these columns but do want the first value for each line
    # so re-get the indices.
    lineIdx = index_containing_substring(channelNames, 'line')
    flightIdx = index_containing_substring(channelNames, 'flight')
    dateIdx = index_containing_substring(channelNames, 'date')
    zoneIdx = index_containing_substring(channelNames, 'zone')
    print(f'Channels for group metadata Indices:')
    print(f'    line {lineIdx}, {channelNames[lineIdx]};')
    print(f'    flight {flightIdx}, {channelNames[flightIdx]};')
    print(f'    date {dateIdx}, {channelNames[dateIdx]};')
    print(f'    zone {zoneIdx}, {"" if zoneIdx < 0 else channelNames[zoneIdx]}.')

    # now if index not found - error/warning; else pop off the channelName list
    if omitChannels == []:
        if lineIdx < 0:
            print('ERROR - lineIdx not found')
            return
        else:
            channelsOut.pop(lineIdx)
            print(f'Line channel = {channelNames[lineIdx]}')
        tempIdx = index_containing_substring(channelsOut, 'flight')
        if flightIdx < 0:
            print('WARNING - flightIdx not found')
        else:
            channelsOut.pop(tempIdx)
            print(f'Flight channel = {channelNames[flightIdx]}')
        tempIdx = index_containing_substring(channelsOut, 'date')
        if dateIdx < 0:
            print('WARNING - dateIdx not found')
        else:
            channelsOut.pop(tempIdx)
            print(f'Date channel = {channelNames[dateIdx]}')
        tempIdx = index_containing_substring(channelsOut, 'zone')
        if zoneIdx < 0:
            print('WARNING - zoneIdx not found')
        else:
            channelsOut.pop(tempIdx)
            print(f'Zone channel = {channelNames[zoneIdx]}')
    else:
        for channel in omitChannels:
            tempIdx = index_containing_substring(channelsOut, channel.lower())
            if tempIdx < 0:
                print(f'WARNING - {channel} to omit not found')
            else:
                channelsOut.pop(tempIdx)
            
    print('Channels out: ')
    print(channelsOut)

    return channelNames, channelsOut, lineIdx, flightIdx, dateIdx, zoneIdx

`lines, line_lengths = get_gdflineinfo(GDF_file, line_chan, field_names)`

In [202]:
def get_gdflineinfo(GDF_file, line_chan, channelNames, flight_chan='', date_chan=''):
    gdf = aseg.read(GDF_file, method='fixed-widths')#, engine='pandas')
    df = gdf.df()
    gby = df.groupby(df[line_chan])
    lines = []
    flights = []
    dates = []
    line_lengths = []
    result = gby.count()
    for i in range(0, len(result)):
        lines.append(result.axes[0][i])
        line_lengths.append(result.iloc[0].values[0])
        # The previous line implicitly assumes all channels have the same number of values.
        # ToDo: Check if this is true.
        # for j in range(0, len(field_names)-1):
        #     print(result[field_names[0]].values[0])#result.axes[1][j])
    
    print(line_chan, flight_chan, date_chan)
    # print(lines)
    for line in lines:
        print(f'    ... Line {line}\n')
        if flight_chan != '':
            # flights.append(df.loc[df[line_chan] == line, flight_chan][0])
            aa = df.loc[df[line_chan] == line, flight_chan].iloc[0]
            flights.append(aa)
            # print(f'    Flights {flights}')
        if date_chan != '':
            dates.append(df.loc[df[line_chan] == line, date_chan].iloc[0])
            # print(f'    Dates {dates}')
        
    return lines, line_lengths, flights, dates

`asegToHDF(datFilePath, outputHdf = '', omitChannels=['RT'], verbose=False)`

In [236]:
def asegToHDF(datFilePath, outputHdf = '', omitChannels=['RT'], verbose=False):
    """
    Reads the data from the ASEG-GDF2 survey file and writes it to a new Whizz
    HDF5 survey file. Uses the aseg_gdf2 package by Kent Inverarity at:
    https://github.com/kinverarity1/aseg_gdf2. Relies on version 0.3!!!

    Parameters
    ----------
    datFilePath : pathlib.PosixPath
        The pathlib Path to the input ASEG-GDF2 file.
    outputHdf : pathlib.PosixPath, optional
        The pathlib Path to the output Whizz HDF5 file. The default is '' and
        the output file is then the same as the input file with the extension
        changed to '.hdf5'.
    omitChannels : [String]
        An array of channel or field names to omit from the saved geoWhizz HDF5 file.
    verbose : Bool
        If true, print extra information whilst saving data.

    Returns
    -------
    None.

    """

    print('\nGetting channel names and indices of key channels ...')
    channelNames, channelsOut, lineIdx, flightIdx, dateIdx, zoneIdx = get_gdfchannames(datFilePath, omitChannels=['RT'])

    print('\n... done. Getting line numbers and line lengths ...')
    if flightIdx >= 0:
        flight_chan = channelsOut[flightIdx-1]
    else:
         flight_chan = ''
    if dateIdx >= 0:
        date_chan = channelsOut[dateIdx-1]
    else:
         date_chan = ''
    line_chan = channelNames[lineIdx-1]
    print(line_chan, flight_chan, date_chan)
    # print(channelNames)
    # print(channelsOut)
    lines, line_lengths, flights, dates = get_gdflineinfo(datFilePath, line_chan, channelNames, flight_chan=flight_chan, date_chan=date_chan)
    print('... done. Ready to create HDF file and transfer data into it.\n')

    if outputHdf == '':
        outputHdf = datFilePath.with_suffix('.hdf5')

    gdf = aseg.read(str(datFilePath), method='fixed-widths')
    mydataframe = gdf.df()
    with h5py.File(str(outputHdf), 'w') as f:
        # create all the data structure ready for the datasets
        g = f.create_group(groupName)
        g.attrs['ProjectName'] = projectName
        
        gCoord = g.create_group('CoordinateFrame')
        gLines = g.create_group('Lines')
        if zoneIdx >= 0:
            # s/w limitation - assume all data in the same UTM zone as the first flight line.
            gCoord.attrs['UTMZone'] = zones[0]
        
        # Create the data structure and store the metadata
        for ii in range(0, len(lines)):
            # create a line group and metadata
            gg = gLines.create_group(f'{lines[ii]}')
            gg.attrs['LineNumber'] = lines[ii]
            if flightIdx >= 0:
                gg.attrs['Flight'] = flights[ii]
            if dateIdx >= 0:
                gg.attrs['Date_Local'] = dates[ii]
            gg.attrs['NumberOfFids'] = line_lengths[ii]

        # Move the data.
        for lineCount in range(0, len(lines)): #for aLine in lines:
            if verbose:
                print('  About to add data for line ', lines[lineCount], ' Lcount ', lineCount+1, '/', len(lines))
            for channelName in channelsOut:
                #print('Field name ', channelsOut[count])
                
                dataArray = mydataframe.loc[mydataframe[line_chan] == lines[lineCount], channelName]
                
                if dataArray.dtype == 'int64':
                    dd = gg.create_dataset(channelName, data = dataArray, dtype='int64', compression="gzip", compression_opts=4)
                elif dataArray.dtype == 'float64':
                    dd = gg.create_dataset(channelName, data = dataArray, dtype='float64', compression="gzip", compression_opts=4)
                else:
                    nparray = np.array(dataArray)
                    dd = gg.create_dataset(channelName, data = nparray, compression="gzip", compression_opts=4)

                dd.attrs['Name'] = channelName
                dd.attrs['Alias'] = channelName
                fieldDef = gdf.get_field_definition(channelName)
                dd.attrs['Units'] = cleanUnits(fieldDef['unit'])
                dd.attrs['Description'] = fieldDef['long_name']

    return

`units_str = cleanUnits(fieldDef)`

In [6]:
def cleanUnits(fieldDef):
    """
    ASEG-GDF2 files generated by Atlas (proprietary geophysical processing software
    from Fugro Airborne, CGG Airborne and XCalibur) does not obey the ASEG-GDF2
    standard. One issue is the use of an extra colon where the standard requires
    a comma, making the units field incorrect. This routine takes a units fieldDef
    and fixes the error by removing the substring from the extra colon. 

    Parameters
    ----------
    fieldDef : String
        The field units from the dictionary fieldDef['units'].

    Returns
    -------
    String
        The field units after correction.

    """
    return fieldDef.split(':')[0]

`my_index = index_containing_substring(the_list, substring)`

In [7]:
def index_containing_substring(the_list, substring):
    """
    Returns the index to the substring in the_list; -1 if not found.

    Parameters
    ----------
    the_list : String
        The string to be searched.
    substring : String
        The substring whose position in the_list is desired.

    Returns
    -------
    Integer
        The index to the start of the substring, -1 if not found.

    """
    lowlist = [name.lower() for name in the_list]
    for i, s in enumerate(lowlist):
        if substring in s:
              return i
    return -1

### Muppet Mag Test 

#### get_gdfchannels works!

In [147]:
channelNames, channelsOut, lineIdx, flightIdx, dateIdx, zoneIdx = get_gdfchannames(muppetGDF_file, omitChannels=['RT'])

Channels for group metadata Indices:
    line 1, LINE;
    flight 2, FLIGHT;
    date 3, DATE;
    zone -1, .
1 channels out: 
['BGS_JOB', 'LINE', 'FLIGHT', 'DATE', 'FIDUCIAL', 'EAST_MGA', 'GDA94LAT', 'GDA94LON', 'MAGUNCMP', 'MAGCOMP', 'DIURNAL', 'IGRF', 'MAG_LEV', 'RAD_ALT', 'GPS_HT', 'DEM']


In [148]:
lineIdx

1

In [149]:
flightIdx

2

In [150]:
dateIdx

3

In [151]:
zoneIdx

-1

In [152]:
channelsOut[flightIdx]

'FLIGHT'

In [153]:
channelsOut[dateIdx]

'DATE'

In [154]:
channelsOut[lineIdx]

'LINE'

In [155]:
channelsOut[0]

'BGS_JOB'

#### get_gdflineinfo works

In [156]:
lines, line_lengths, flights, dates = get_gdflineinfo(muppetGDF_file, channelsOut[lineIdx], lineIdx, ['FIDUCIAL'], flight_chan=channelsOut[flightIdx], 
                                                      date_chan=channelsOut[dateIdx])

LINE FLIGHT DATE
[10010.0]
    ... Line 10010.0

    Flights [1.0]
    Dates [20091202.0]


In [157]:
lines

[10010.0]

In [158]:
line_lengths

[1050]

In [159]:
flights

[1.0]

In [160]:
dates

[20091202.0]

#### asegToHDF works (1 line file)!

In [219]:
asegToHDF(muppetGDF_file, outputHdf = '', omitChannels=['RT'], verbose=False)


Getting channel names and indices of key channels ...
Channels for group metadata Indices:
    line 1, LINE;
    flight 2, FLIGHT;
    date 3, DATE;
    zone -1, .
Channels out: 
['BGS_JOB', 'LINE', 'FLIGHT', 'DATE', 'FIDUCIAL', 'EAST_MGA', 'GDA94LAT', 'GDA94LON', 'MAGUNCMP', 'MAGCOMP', 'DIURNAL', 'IGRF', 'MAG_LEV', 'RAD_ALT', 'GPS_HT', 'DEM']

... done. Getting line numbers and line lengths ...
BGS_JOB LINE FLIGHT
BGS_JOB LINE FLIGHT
    ... Line 954

... done. Ready to create HDF file and transfer data into it.



### Canobie Mag Test 

In [177]:
canobieGDF_file = Path(r'/Volumes/T7/FalconSurveys/202111_Falcon_Canobie_GA/2022-03-02-FinalsRevised/located/902212_AGG.dfn')

#### get_gdfchannames works (many line file)

In [163]:
channelNames, channelsOut, lineIdx, flightIdx, dateIdx, zoneIdx = get_gdfchannames(canobieGDF_file, omitChannels=['RT'])

Channels for group metadata Indices:
    line 1, FLTLINE;
    flight 45, Flight;
    date 5, Date;
    zone -1, .
1 channels out: 
['FLTLINE', 'Altitude_Geoid09', 'Altitude_GRS80', 'Bearing', 'Date', 'DTM', 'Falcon_ANE_ANR_0p0', 'Falcon_ANE_ANR_2p67', 'Falcon_ANE_Measured_0p0', 'Falcon_ANE_Measured_2p67', 'Falcon_AUV_ANR_0p0', 'Falcon_AUV_ANR_2p67', 'Falcon_AUV_Measured_0p0', 'Falcon_AUV_Measured_2p67', 'Falcon_BNE_ANR_0p0', 'Falcon_BNE_ANR_2p67', 'Falcon_BNE_Measured_0p0', 'Falcon_BNE_Measured_2p67', 'Falcon_BUV_ANR_0p0', 'Falcon_BUV_ANR_2p67', 'Falcon_BUV_Measured_0p0', 'Falcon_BUV_Measured_2p67', 'Falcon_gD_0p0', 'Falcon_gD_0p0_HP', 'Falcon_gD_2p67', 'Falcon_gD_2p67_HP', 'Falcon_GDD_0p0', 'Falcon_GDD_2p67', 'Falcon_GED_0p0', 'Falcon_GED_2p67', 'Falcon_GEE_0p0', 'Falcon_GEE_2p67', 'Falcon_GND_0p0', 'Falcon_GND_2p67', 'Falcon_GNE_0p0', 'Falcon_GNE_2p67', 'Falcon_GNN_0p0', 'Falcon_GNN_2p67', 'Falcon_GUV_0p0', 'Falcon_GUV_2p67', 'Falcon_NE_ANR_Noise', 'Falcon_NE_Measured_Noise', 'Falcon

In [164]:
channelNames[0], len(channelNames)

('FLTLINE', 79)

In [165]:
channelsOut[0], len(channelsOut)

('FLTLINE', 79)

In [166]:
lineIdx

1

In [167]:
flightIdx

45

In [168]:
dateIdx

5

In [169]:
zoneIdx

-1

In [170]:
channelsOut[lineIdx-1]

'FLTLINE'

In [171]:
channelsOut[flightIdx-1]

'Flight'

In [172]:
channelsOut[dateIdx-1]

'Date'

#### get_gdflineinfo works (many line file)?

In [173]:
lines, line_lengths, flights, dates = get_gdflineinfo(canobieGDF_file, 'FLTLINE', lineIdx-1, ['UTC_Local'], flight_chan=channelsOut[44], 
                                                      date_chan=channelsOut[4])

FLTLINE Flight Date
[100010.0, 100020.0, 100030.0, 100040.0, 100050.0, 100060.0, 100070.0, 100080.0, 100090.0, 100100.0, 100110.0, 100120.0, 100130.0, 100140.0, 100151.0, 100160.0, 100170.0, 100180.0, 100190.0, 100200.0, 100210.0, 100220.0, 100230.0, 100240.0, 100250.0, 100260.0, 100271.0, 100280.0, 100290.0, 100300.0, 100310.0, 100320.0, 100330.0, 100340.0, 100351.0, 100360.0, 100371.0, 100381.0, 100390.0, 100400.0, 100410.0, 100420.0, 100430.0, 100440.0, 100450.0, 100460.0, 100470.0, 100480.0, 100490.0, 100500.0, 100510.0, 100520.0, 100530.0, 100540.0, 100550.0, 100560.0, 100570.0, 100580.0, 100590.0, 100600.0, 100610.0, 100620.0, 100630.0, 100640.0, 100650.0, 100660.0, 100670.0, 100680.0, 100690.0, 100700.0, 100710.0, 100720.0, 100730.0, 100740.0, 100750.0, 100760.0, 100770.0]
    ... Line 100010.0

    Flights [2]
    Dates ['2021/11/24']
    ... Line 100020.0

    Flights [2, 2]
    Dates ['2021/11/24', '2021/11/24']
    ... Line 100030.0

    Flights [2, 2, 3]
    Dates ['2021/11

In [ ]:
lines

In [ ]:
line_lengths

In [174]:
flights

[2,
 2,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 10,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 10,
 6,
 10,
 10,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 10,
 10,
 10,
 10,
 10]

In [ ]:
dates

#### asegToHDF works (many line file)?

In [191]:
gdf = aseg.read(canobieGDF_file, method='fixed-widths')#, engine='pandas')
# channelNames = gdf.field_names()

In [ ]:
import AirGravQC as qc


In [237]:
asegToHDF(canobieGDF_file, outputHdf = '', omitChannels=['RT'], verbose=True)


Getting channel names and indices of key channels ...
Channels for group metadata Indices:
    line 1, FLTLINE;
    flight 45, Flight;
    date 5, Date;
    zone -1, .
Channels out: 
['FLTLINE', 'Altitude_Geoid09', 'Altitude_GRS80', 'Bearing', 'Date', 'DTM', 'Falcon_ANE_ANR_0p0', 'Falcon_ANE_ANR_2p67', 'Falcon_ANE_Measured_0p0', 'Falcon_ANE_Measured_2p67', 'Falcon_AUV_ANR_0p0', 'Falcon_AUV_ANR_2p67', 'Falcon_AUV_Measured_0p0', 'Falcon_AUV_Measured_2p67', 'Falcon_BNE_ANR_0p0', 'Falcon_BNE_ANR_2p67', 'Falcon_BNE_Measured_0p0', 'Falcon_BNE_Measured_2p67', 'Falcon_BUV_ANR_0p0', 'Falcon_BUV_ANR_2p67', 'Falcon_BUV_Measured_0p0', 'Falcon_BUV_Measured_2p67', 'Falcon_gD_0p0', 'Falcon_gD_0p0_HP', 'Falcon_gD_2p67', 'Falcon_gD_2p67_HP', 'Falcon_GDD_0p0', 'Falcon_GDD_2p67', 'Falcon_GED_0p0', 'Falcon_GED_2p67', 'Falcon_GEE_0p0', 'Falcon_GEE_2p67', 'Falcon_GND_0p0', 'Falcon_GND_2p67', 'Falcon_GNE_0p0', 'Falcon_GNE_2p67', 'Falcon_GNN_0p0', 'Falcon_GNN_2p67', 'Falcon_GUV_0p0', 'Falcon_GUV_2p67', 'Falc

ValueError: Unable to create dataset (name already exists)

In [221]:
gdf = aseg.read(canobieGDF_file, method='fixed-widths')#, engine='pandas')
df = gdf.df()
gby = df.groupby(df['FLTLINE'])
lines = []
flights = []
dates = []
line_lengths = []
result = gby.count()

In [198]:
result

,RT,Altitude_Geoid09,Altitude_GRS80,Bearing,Date,DTM,Falcon_ANE_ANR_0p0,Falcon_ANE_ANR_2p67,Falcon_ANE_Measured_0p0,Falcon_ANE_Measured_2p67,...,Regional_Grav_Freeair_LP,Supplier_Job_Number,T_1p0_DD,T_1p0_NE,T_1p0_UV,Turbulence,UTC_1980,UTC_Local,X_GDA2020,Y_GDA2020
FLTLINE,,,,,,,,,,,,,,,,,,,,,
100010.0,8201,8201,8201,8201,8201,8201,8201,8201,8201,8201,...,8201,8201,8201,8201,8201,8201,8201,8201,8201,8201
100020.0,8265,8265,8265,8265,8265,8265,8265,8265,8265,8265,...,8265,8265,8265,8265,8265,8265,8265,8265,8265,8265
100030.0,8161,8161,8161,8161,8161,8161,8161,8161,8161,8161,...,8161,8161,8161,8161,8161,8161,8161,8161,8161,8161
100040.0,8393,8393,8393,8393,8393,8393,8393,8393,8393,8393,...,8393,8393,8393,8393,8393,8393,8393,8393,8393,8393
100050.0,8241,8241,8241,8241,8241,8241,8241,8241,8241,8241,...,8241,8241,8241,8241,8241,8241,8241,8241,8241,8241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100730.0,8465,8465,8465,8465,8465,8465,8465,8465,8465,8465,...,8465,8465,8465,8465,8465,8465,8465,8465,8465,8465
100740.0,8265,8265,8265,8265,8265,8265,8265,8265,8265,8265,...,8265,8265,8265,8265,8265,8265,8265,8265,8265,8265
100750.0,8393,8393,8393,8393,8393,8393,8393,8393,8393,8393,...,8393,8393,8393,8393,8393,8393,8393,8393,8393,8393


In [201]:
result.iloc[0].values[0]

8201

In [222]:
dataArray = df.loc[df['FLTLINE'] == 100010.0, 'Date']
if dataArray.dtype == 'int64':
    print('Int')

In [223]:
dataArray

0       2021/11/24
1       2021/11/24
2       2021/11/24
3       2021/11/24
4       2021/11/24
           ...    
8196    2021/11/24
8197    2021/11/24
8198    2021/11/24
8199    2021/11/24
8200    2021/11/24
Name: Date, Length: 8201, dtype: object

In [225]:
bb = dataArray.iloc[0]

In [226]:
bb

'2021/11/24'

In [227]:
type(bb)

str

In [228]:
str(1)

'1'

In [231]:
xx = np.array(dataArray)

In [232]:
xx

array(['2021/11/24', '2021/11/24', '2021/11/24', ..., '2021/11/24',
       '2021/11/24', '2021/11/24'], dtype=object)

In [233]:
xx[0]

'2021/11/24'

In [238]:
import gspy

In [239]:
import matplotlib.pyplot as plt
from os.path import join
from gspy import Survey

In [242]:
# Path to example files
# data_path = '..//..//supplemental//region//MAP'
# canobieGDF_file = Path(r'/Volumes/T7/FalconSurveys/202111_Falcon_Canobie_GA/2022-03-02-FinalsRevised/located/902212_AGG.dfn')
# Survey Metadata file
metadata = r'/Volumes/T7/FalconSurveys/202111_Falcon_Canobie_GA/2022-03-02-FinalsRevised/located/902212_AGG.json'#join(data_path, "data//Tempest_survey_md.json")

# Establish survey instance
survey = Survey(metadata)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)